In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import datasets, transforms

In [9]:
import time

In [10]:
class Preprocessor:

    def __init__(self, conf):
        self.conf = conf
    
    def create_transforms(self, data_transforms):
        transforms_list = []
        for trans in data_transforms:
            if trans["type"] == "elastic":
                transforms_list.append(transforms.ElasticTransform())
                print("Elastic transformation applied.")
            if trans["type"] == "rotation":
                transforms_list.append(transforms.RandomRotation(trans["degree"]))
                print("Rotation transformation applied.")
            if trans["type"] == "pad":
                transforms_list.append(transforms.Pad(trans["padding"]))
                print("Padding transformation applied.")
            if trans["type"] == "resize":
                transforms_list.append(transforms.Resize(trans["size"]))
                print("Resize transformation applied.")
        return transforms_list
        

    def load(self, transforms_list, train_bool):

        transforms_list.append(transforms.ToTensor())
               
        if self.conf["normalization"] == True:
            transforms_list.append(transforms.Normalize((0.1307,), (0.3081,)))
        
        transform = transforms.Compose(transforms_list)

        dataset = datasets.MNIST(root='./data', train=train_bool, download=True, transform=transform)
        return DataLoader(dataset, batch_size=self.conf["batch_size"], shuffle=train_bool, generator=torch.Generator().manual_seed(self.conf["seed"]))

In [11]:
class TrainDataset:
    
    def __init__(self, conf):
        self.conf = conf
        self.data_loader = None
        self.preprocessor = Preprocessor(self.conf)

    def preprocess(self):
        transforms_list = []
        transforms_list += self.preprocessor.create_transforms(self.conf["data_transforms"])
        transforms_list += self.preprocessor.create_transforms(self.conf["train_data_transforms"])
        self.data_loader = self.preprocessor.load(transforms_list, True)

In [12]:
class TestDataset:
    
    def __init__(self, conf):
        self.conf = conf
        self.data_loader = None
        self.preprocessor = Preprocessor(self.conf)

    def preprocess(self):
        transforms_list = []
        transforms_list += self.preprocessor.create_transforms(self.conf["data_transforms"])
        transforms_list += self.preprocessor.create_transforms(self.conf["test_data_transforms"])
        self.data_loader = self.preprocessor.load(transforms_list, False)

In [21]:
class Net(nn.Module):
    def __init__(self, conf):
        super(Net, self).__init__()
        self.conf = conf
        self.net = nn.ModuleList()
        
        layers = self.conf["layers"]

        for layer in layers:
            if layer["type"] == "conv2d":
                dilation = 1
                padding = 0
                if (layer.get("dilation") != None):
                    dilation = layer["dilation"]
                if (layer.get("padding") != None):
                    padding = layer["padding"]
                self.net.append(nn.Conv2d(layer["input_channels"], layer["output_channels"], layer["kernel_size"], dilation=dilation, padding=padding).to(self.conf["device"]))

            elif layer["type"] == "fc":
                self.net.append(nn.Linear(layer["input_channels"], layer["output_channels"]).to(self.conf["device"]))
            
            elif layer["type"] == "dropout":
                self.net.append(nn.Dropout()).to(self.conf["device"])

    def forward(self, x):
        layers = self.conf["layers"]
        i = 0
        for layer in layers:
            if layer["type"] == "conv2d":
                x = self.net[i](x)
                i = i + 1
            
            elif layer["type"] == "avg_pool2d":
                x = F.avg_pool2d(x, (2, 2))

            elif layer["type"] == "max_pool2d":
                x = F.max_pool2d(x, (2, 2))
            
            elif layer["type"] == "fc":
                # si x contient des images (n'est pas un vecteur)
                if len(x.shape) > 2:
                    x = x.view(-1, layer["input_channels"])
                x = self.net[i](x)
                i = i + 1

            elif layer["type"] == "dropout":
                x = self.net[i](x)
                i = i + 1

            # activation function (optional)
            act = layer.get("activation")
            if act == "tanh":
                x = torch.tanh(x)
            elif act == "relu":
                x = F.relu(x)
            elif act == "softmax":
                x = F.softmax(x, dim=1)

        return x

    def display(self):
        print(self)
        return


In [14]:
class Trainer:

    def __init__(self, net, dataset, conf):

        self.net = net.to(conf["device"])
        self.dataset = dataset
        self.conf = conf

    def train(self):

        # pour l'initialisation des paramètres
        torch.manual_seed(self.conf["seed"])

        # ensure model in training mode
        self.net.train()

        # initialisation des paramètres d'apprentissage

        if self.conf["loss_function"]["type"] == "cross_entropy":
            criterion = nn.CrossEntropyLoss()

        if self.conf["optimizer"]["type"] == "sgd":
            optimizer = optim.SGD(self.net.parameters(), lr=self.conf["optimizer"]["lr"], momentum=self.conf["optimizer"]["momentum"])

        if self.conf["optimizer"]["type"] == "adam":
            optimizer = optim.Adam(self.net.parameters(), lr=self.conf["optimizer"]["lr"])
        
        num_epochs = self.conf["num_epochs"]
        # loop over the dataset multiple times
        for epoch in range(num_epochs):

            # pour chaque batch
            for i, data in enumerate(self.dataset.data_loader, 0):
                # get the inputs
                inputs, labels = data
                inputs = inputs.to(self.conf["device"])
                labels = labels.to(self.conf["device"])

                # remettre les gradients (un gradient pour chaque paramètre) à zéro
                optimizer.zero_grad()

                # calcul des sorties du réseau pour le batch entier
                outputs = self.net(inputs)
                # calcul de la perte
                loss = criterion(outputs, labels)
                # calcul des gradients par rapport à tous les paramètres du réseau
                loss.backward()
                # ajustement des poids en fonction des gradients
                optimizer.step()

                # print statistics
                mod = len(self.dataset.data_loader.dataset) * self.conf["num_epochs"] // self.conf["batch_size"] // 10
                if i % mod == 0 :    # print for 10 mini-batches
                    print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, loss.item()))
                    
        print('Finished Training')

In [16]:
class Evaluator :

    def __init__(self, net, dataset, conf):

        self.net = net.to(conf["device"])
        self.dataset = dataset
        self.conf = conf
    
    def evaluate(self):

        # ensure model in evaluation mode (disables Dropout, sets BatchNorm to eval)
        self.net.eval()
        
        correct = 0
        total = 0        
        # accumulate separate timings
        total_load_time = 0.0
        total_forward_time = 0.0
        
        # désactivation du calcul des gradients
        with torch.no_grad():
            # pour chaque batch
            batch_load_start = time.perf_counter()
            for data in self.dataset.data_loader:
                images, labels = data
                images = images.to(self.conf["device"])
                labels = labels.to(self.conf["device"])
                after_to_device = time.perf_counter()

                # time spent loading/preprocessing for this batch (including transforms and to(device))
                total_load_time += (after_to_device - batch_load_start)

                # forward pass
                outputs = self.net(images)
                after_forward = time.perf_counter()

                # forward pass timing
                total_forward_time += (after_forward - after_to_device)

                # récupérer les indices des classes prédites
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

                batch_load_start = time.perf_counter()

        print('Finished Testing')
        print(f"Accuracy of the network on {total} data points : {100 * correct / total}%")
        print(f"Inference time (forward): {total_forward_time:.3f}s, Loading time: {total_load_time:.3f}s")
        

    def display(self):

        self.net.eval()

        y_true = []
        y_pred = []

        with torch.no_grad():
            # pour chaque batch
            for data in self.dataset.data_loader:
                images, labels = data
                images = images.to(self.conf["device"])
                labels = labels.to(self.conf["device"])

                outputs = self.net(images)

                # récupérer les indices des classes prédites
                _, predicted = torch.max(outputs.data, 1)
                
                # accumuler les vraies étiquettes et les prédictions
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(predicted.cpu().numpy())

        if "confusion_matrix" in self.conf["display"]:
            classes = self.dataset.data_loader.dataset.classes
            cm = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
            disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['0','1','2','3','4','5','6','7','8','9'])
            disp.plot(cmap=plt.cm.Blues)
            plt.title('Confusion Matrix', fontsize=15, pad=20)
            plt.xlabel('Prediction', fontsize=11)
            plt.ylabel('Actual', fontsize=11)
            plt.gca().xaxis.set_label_position('top')
            plt.gca().xaxis.tick_top()
            plt.gca().figure.subplots_adjust(bottom=0.2)
            plt.gca().figure.text(0.5, 0.05, 'Prediction', ha='center', fontsize=13)
            
        return

In [17]:
class TrainDatasetFactory:
    @staticmethod
    def create(conf):
        if conf["dataset"] == "mnist":
            return TrainDataset(conf)
        
class TestDatasetFactory:
    @staticmethod
    def create(conf):
        if conf["dataset"] == "mnist":
            return TestDataset(conf)

class NetFactory:
    @staticmethod
    def create(conf):
        return Net(conf)
        
class TrainerFactory:
    @staticmethod
    def create(net, dataset, conf):
            return Trainer(net, dataset, conf)
    
class EvaluatorFactory:
    @staticmethod
    def create(net, dataset, conf):
            return Evaluator(net, dataset, conf)

In [18]:
class Experiment:

    def __init__(self, conf):
        self.conf = conf
        self.train_dataset = None
        self.test_dataset = None
        self.net = None
        self.trainer = None
        self.evaluator = None

    def prepare_train_data(self):
        self.train_dataset = TrainDatasetFactory.create(self.conf)
        self.train_dataset.preprocess()
    
    def prepare_test_data(self):
        self.test_dataset = TestDatasetFactory.create(self.conf)
        self.test_dataset.preprocess()

    def build_network(self):
        self.net = NetFactory.create(self.conf)
        self.net.display()

    def train(self):
        self.trainer = TrainerFactory.create(self.net, self.train_dataset, self.conf)
        self.trainer.train()

    def test(self):
        if self.conf["inference_data"] == "train":
            self.evaluator = EvaluatorFactory.create(self.net, self.train_dataset, self.conf)
        if self.conf["inference_data"] == "test":
            self.evaluator = EvaluatorFactory.create(self.net, self.test_dataset, self.conf)
        self.evaluator.evaluate()

    def display_result(self):
        self.evaluator.display()

    def run(self):
        self.prepare_train_data()
        self.prepare_test_data()
        self.build_network()
        self.train()
        self.test()

In [19]:
device = torch.device('cpu')  # Force l'utilisation du CPU

In [88]:
conf = {
    # device configuration
    "device": device,

    # seed configuration (for data shuffle and weights initialisation)
    "seed": 42,
    
    # data configuration
    "dataset": "mnist",
    "normalization": True,
    "data_transforms": [
        {"type": "pad", "padding": 2}
    ],

    # net configuration
    "layers": [
        {"type": "conv2d", "input_channels": 1, "output_channels": 20, "kernel_size": 5, "activation": "relu"},
        {"type": "max_pool2d", "kernel_size": 2},
        {"type": "conv2d", "input_channels": 20, "output_channels": 40, "kernel_size": 5, "activation": "relu"},
        {"type": "max_pool2d", "kernel_size": 2},
        {"type": "fc", "input_channels": 40*4*4, "output_channels": 640, "activation": "relu"},
        {"type": "dropout1d"},
        {"type": "fc", "input_channels": 640, "output_channels": 1000, "activation": "relu"},
        {"type": "dropout1d"},
        {"type": "fc", "input_channels": 1000, "output_channels": 10, "activation": "softmax"}
    ],
    
    # train configuration
    "train_data_transforms": [],
    "loss_function": {"type": "cross_entropy"},
    "optimizer": {"type": "sgd", "lr": 0.005, "momentum": 0.9},
    "num_epochs": 10,
    "batch_size": 32,
    
    # test configuration
    "test_data_transforms": [],
    "inference_data": "test",
    "display": ["confusion_matrix"]
}

In [82]:
experiment = Experiment(conf)

In [83]:
experiment.prepare_train_data()

Padding transformation applied.


In [84]:
experiment.prepare_test_data()

Padding transformation applied.


In [85]:
experiment.build_network()

Net(
  (net): ModuleList(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
    (1): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2), dilation=(2, 2))
    (2): Linear(in_features=400, out_features=120, bias=True)
    (3): Linear(in_features=120, out_features=84, bias=True)
    (4): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [86]:
experiment.train()

[1,     1] loss: 2.302
[2,     1] loss: 1.659
[3,     1] loss: 1.633
[4,     1] loss: 1.624
[5,     1] loss: 1.566
[6,     1] loss: 1.617
[7,     1] loss: 1.585
[8,     1] loss: 1.616
[9,     1] loss: 1.641
[10,     1] loss: 1.511
Finished Training


In [87]:
experiment.test()

Finished Testing
Accuracy of the network on 10000 data points : 97.92%
Inference time (forward): 0.331s, Loading time: 1.871s


In [69]:
experiment.display_result()

AttributeError: 'NoneType' object has no attribute 'display'